# 09 — Projection Plugins

The 0.3.5 release added a plugin system for **RDF-declared,
persistent** projection pipelines — the counterpart to the ephemeral
Python-only pipelines shown in `04_projections`.

Pipelines are modeled as `cga:ProjectionPipelineSpec` resources in
the registry graph. Python transforms are discovered via the
`holonic.projections` entry-point group (both first-party and
third-party).

This notebook covers:

- The `@projection_transform` decorator for registering transforms
- Building a `ProjectionPipelineSpec` from Python dataclasses
- `register_pipeline()`, `attach_pipeline()`, `list_pipelines()`
- `run_projection()` and the provenance it records
- Third-party transforms registered at runtime
- The Turtle escape hatch via `register_pipeline_ttl()`


## Setup


In [ ]:
from holonic import (
    HolonicDataset,
    ProjectionPipelineSpec,
    ProjectionPipelineStep,
    get_registered_transforms,
    projection_transform,
)
from rdflib import Graph

ds = HolonicDataset()

ds.add_holon("urn:holon:data", "Data Holon")
ds.add_interior("urn:holon:data", '''
    @prefix ex: <urn:ex:> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

    <urn:item:a> a ex:Widget ; rdfs:label "alpha" .
    <urn:item:b> a ex:Widget ; rdfs:label "beta" .
    <urn:item:c> a ex:Gadget ; rdfs:label "gamma" .
    _:blank a ex:Hidden .
''')


## First-party transforms are discoverable

Three transforms ship in `holonic.projections`:
`strip_blank_nodes`, `localize_predicates`, `collapse_reification`.
They register via the same entry-point group third parties use
(dogfooding).


In [ ]:
registry = get_registered_transforms()
for name in sorted(registry):
    print(f"  {name}: {registry[name].__module__}.{registry[name].__name__}")


## Third-party transforms: register at runtime

Any function matching `(Graph) -> Graph` can be registered via the
`@projection_transform` decorator. Third-party packages declare them
in their own `pyproject.toml` `[project.entry-points."holonic.projections"]`
section; in-process registration is equivalent for dev and notebook use.


In [ ]:
@projection_transform("uppercase_labels")
def uppercase_labels(g: Graph) -> Graph:
    from rdflib import Literal
    from rdflib.namespace import RDFS
    out = Graph()
    for s, p, o in g:
        if p == RDFS.label and isinstance(o, Literal):
            out.add((s, p, Literal(str(o).upper())))
        else:
            out.add((s, p, o))
    return out

print(f"registered: {list(get_registered_transforms())}")


## Build and register a pipeline

A `ProjectionPipelineSpec` is a Python dataclass; `register_pipeline()`
serializes it into the registry graph as `cga:ProjectionPipelineSpec`
+ `cga:ProjectionPipelineStep` resources.


In [ ]:
spec = ProjectionPipelineSpec(
    iri="urn:pipeline:viz-basic",
    name="Viz basic",
    description="Strip blanks, uppercase labels, localize predicates.",
    steps=[
        ProjectionPipelineStep(name="strip", transform_name="strip_blank_nodes"),
        ProjectionPipelineStep(name="upper", transform_name="uppercase_labels"),
        ProjectionPipelineStep(name="local", transform_name="localize_predicates"),
    ],
)
ds.register_pipeline(spec)


## Attach to a holon and list

Pipelines are attached to specific holons via `cga:hasPipeline`. One
holon can have many pipelines available; one pipeline can be attached
to many holons.


In [ ]:
ds.attach_pipeline("urn:holon:data", "urn:pipeline:viz-basic")

for summary in ds.list_pipelines("urn:holon:data"):
    print(f"  {summary.iri}")
    print(f"    name: {summary.name}")
    print(f"    steps: {summary.step_count}")
    print(f"    desc: {summary.description}")


## Read the pipeline back

`get_pipeline(iri)` walks the `rdf:List` of steps and returns a
fully-populated `ProjectionPipelineSpec`, preserving step order.


In [ ]:
readback = ds.get_pipeline("urn:pipeline:viz-basic")
print(f"name: {readback.name}")
for s in readback.steps:
    print(f"  {s.name}  [{s.transform_name}]")


## Run the projection

`run_projection()` merges the holon's interior graphs, executes each
step in order, optionally stores the result as a projection layer
graph, and records a `prov:Activity` in the holon's context graph.


In [ ]:
result = ds.run_projection(
    "urn:holon:data",
    "urn:pipeline:viz-basic",
    store_as="urn:holon:data/projection/viz",
    agent_iri="urn:agent:operator",
)

print(f"result: {len(result)} triples")
for s, p, o in sorted(result):
    print(f"  {s}  {p}  {o}")


## Inspect the provenance activity

The activity carries loose version info for each transform used
(`cga:transformVersion`) plus host metadata (`cga:runHost`,
`cga:runPlatform`, `cga:runPythonVersion`, `cga:runHolonicVersion`)
for reproducibility auditing.


In [ ]:
rows = ds.query('''
    PREFIX prov: <http://www.w3.org/ns/prov#>
    PREFIX cga:  <urn:holonic:ontology:>
    SELECT ?activity ?start ?end ?host ?pyver
    WHERE {
        GRAPH ?g {
            ?activity a prov:Activity ;
                prov:used <urn:pipeline:viz-basic> ;
                prov:startedAtTime ?start ;
                prov:endedAtTime ?end ;
                cga:runHost ?host ;
                cga:runPythonVersion ?pyver .
        }
    }
    ORDER BY DESC(?start)
''')
for r in rows:
    print(f"  activity: {r['activity']}")
    print(f"    started: {r['start']}")
    print(f"    host:    {r['host']}")
    print(f"    python:  {r['pyver']}")


## Metadata auto-refreshes on the output graph

Because `run_projection` ran with `store_as="urn:holon:data/projection/viz"`
and the default `metadata_updates="eager"`, the output graph now
carries current metadata.


In [ ]:
md = ds.get_graph_metadata("urn:holon:data/projection/viz")
print(f"output graph triples: {md.triple_count}")
print(f"last modified:        {md.last_modified}")


## Turtle escape hatch

For full control over spec shape (useful when migrating existing
registries), `register_pipeline_ttl(turtle)` accepts raw Turtle.


In [ ]:
ds.register_pipeline_ttl('''
    @prefix cga:  <urn:holonic:ontology:> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

    <urn:pipeline:raw> a cga:ProjectionPipelineSpec ;
        rdfs:label "Raw TTL pipeline" ;
        rdfs:comment "Written directly as Turtle." ;
        cga:hasStep ( <urn:pipeline:raw/step/0> ) .

    <urn:pipeline:raw/step/0> a cga:ProjectionPipelineStep ;
        cga:stepName "strip blanks" ;
        cga:transformName "strip_blank_nodes" .
''')

got = ds.get_pipeline("urn:pipeline:raw")
print(f"{got.name}: {len(got.steps)} step(s)")


## CONSTRUCT-only steps

Pipeline steps can carry a `construct_query` instead of (or alongside)
a `transform_name`. The CONSTRUCT runs against the step's input graph.


In [ ]:
spec = ProjectionPipelineSpec(
    iri="urn:pipeline:widgets-only",
    name="Widgets only",
    steps=[
        ProjectionPipelineStep(
            name="filter widgets",
            construct_query='''
                PREFIX ex: <urn:ex:>
                CONSTRUCT { ?s ?p ?o }
                WHERE { ?s a ex:Widget . ?s ?p ?o }
            ''',
        )
    ],
)
ds.register_pipeline(spec)
ds.attach_pipeline("urn:holon:data", "urn:pipeline:widgets-only")

result = ds.run_projection("urn:holon:data", "urn:pipeline:widgets-only")
for s, p, o in sorted(result):
    print(f"  {s}  {p}  {o}")


## Where to go next

- `04_projections` — the ephemeral Python-only pipeline API, useful
  for one-off processing
- `08_scope_resolution` — find holons to run a pipeline against by
  predicate matching
- `docs/MIGRATION.md` — migration notes when updating pipeline
  definitions across releases
